# PUMA V13.2 — 02 Train Stage 2

Use **50 epochs** to screen all four controlled experiments. After selecting the winner,
set `STAGE2_EPOCHS = 100` and run only that winner from scratch. The curriculum split
stays exactly 30/30/40: 15/15/20 at 50 epochs and 30/30/40 at 100 epochs.

Stage 2 needs the UNI2-h checkpoint, so a Hugging Face token must be reachable
(`HF_TOKEN` in the environment, or the cached `huggingface_hub` login).


In [ ]:
# Project bootstrap. Runs on the "SymbioPan (uv .venv)" kernel on this workstation, and
# on Colab without changes. No %pip here: dependencies come from setup_local.sh (uv) or,
# on Colab, from `!pip install -q -r requirements_colab.txt` in a scratch cell.
from pathlib import Path
import os
import sys

try:
    import google.colab  # type: ignore
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/Research/PUMA')
    ON_COLAB = True
except ImportError:
    PROJECT_DIR = Path.cwd().resolve()
    ON_COLAB = False

PROJECT_DIR = PROJECT_DIR.expanduser().resolve()
if not (PROJECT_DIR / 'puma').is_dir():
    raise RuntimeError(
        f"{PROJECT_DIR} is not the project root (no puma/ package here). "
        "Start JupyterLab from the project root, or set PROJECT_DIR explicitly."
    )
os.chdir(PROJECT_DIR)

# Drop any stale puma modules so an edited package is always re-imported.
for module_name in [m for m in sys.modules if m == 'puma' or m.startswith('puma.')]:
    del sys.modules[module_name]
if str(PROJECT_DIR) in sys.path:
    sys.path.remove(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('PUMA_STAGE2_CROP_CACHE_MB', '512')
os.environ.setdefault('PUMA_V132_AUTO_OOM_FALLBACK', '1')

# GPU selection. On a workstation with two or more GPUs this pins training to GPU 1,
# leaving GPU 0 for the display and for other jobs; with a single GPU (or on Colab) it
# falls back to GPU 0. It must happen before torch is imported, because
# CUDA_VISIBLE_DEVICES is only read when the CUDA driver initialises -- puma.gpu imports
# no torch for that reason. An existing CUDA_VISIBLE_DEVICES is respected, so
#     CUDA_VISIBLE_DEVICES=0 ./.venv/bin/jupyter lab
# still overrides this. The selected GPU becomes cuda:0 inside torch.
from puma.gpu import describe_selection, select_cuda_device

PREFERRED_GPU_INDEX = 1
gpu_selection = select_cuda_device(PREFERRED_GPU_INDEX)

print('PROJECT_DIR =', PROJECT_DIR)
print('python      =', sys.executable)
print()
print(describe_selection(gpu_selection))


In [ ]:
# Verify the kernel is the uv venv and every dependency imports.
# Dependencies are managed by uv, not by %pip:
#     uv pip install -r requirements_colab.txt      (inside .venv)
import importlib
import sys
from pathlib import Path

if not ON_COLAB:
    expected = (PROJECT_DIR / '.venv' / 'bin' / 'python').resolve()
    if Path(sys.executable).resolve() != expected:
        raise RuntimeError(
            f"Wrong kernel: {sys.executable}\nExpected: {expected}\n"
            "In JupyterLab pick Kernel > Change Kernel > 'SymbioPan (uv .venv)'."
        )

for name in ('numpy', 'pandas', 'scipy', 'tifffile', 'shapely', 'rasterio',
             'torch', 'timm', 'huggingface_hub', 'safetensors', 'tqdm', 'psutil'):
    module = importlib.import_module(name)
    print(f"{name:16s} {getattr(module, '__version__', 'unknown')}")

import torch

print('\ncuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    if torch.cuda.device_count() != 1:
        raise RuntimeError(
            f"Expected exactly one visible GPU after selection, saw "
            f"{torch.cuda.device_count()}. CUDA_VISIBLE_DEVICES="
            f"{os.environ.get('CUDA_VISIBLE_DEVICES')!r}. Restart the kernel and Run All."
        )
    properties = torch.cuda.get_device_properties(0)
    physical = gpu_selection.get('selected_index')
    print(f"cuda:0 = physical GPU {physical}: {properties.name}  "
          f"{properties.total_memory / 1024**3:.1f} GB  "
          f"bf16={torch.cuda.is_bf16_supported()}")
    expected_name = gpu_selection.get('selected_name')
    if expected_name and expected_name != properties.name:
        raise RuntimeError(
            f"Selected GPU {physical} is '{expected_name}' in nvidia-smi but torch sees "
            f"'{properties.name}'. The device selection did not take effect; restart the "
            "kernel and Run All."
        )


In [ ]:
from puma.runtime import create_runtime, preflight_environment

# ---- USER SWITCH ----
STAGE2_EPOCHS = 50  # allowed: 50 (screening) or 100 (winner rerun)
WINNER_EXPERIMENT = 'V13_2_02_META_RARE_BS'  # replace after reviewing the 50-epoch results
FAST_NONDETERMINISTIC = True  # faster CUDA kernels; set False for strict reproducibility
STAGE2_DATALOADER_WORKERS = 4

runtime = create_runtime(
    PROJECT_DIR,
    run_folds=(0, 1, 2, 3, 4),
    seeds=(0,),
    stage1_epochs=40,
    stage2_epochs=STAGE2_EPOCHS,
    stage1_effective_batch_size=16,
    stage2_effective_batch_size=256,
    stage1_micro_batch_size=16,
    stage2_micro_batch_size=256,
    preprocessing_workers=0,  # 0 = all logical CPU cores
    early_stopping_enabled=False,
    early_stopping_patience=15,
    early_stopping_min_delta=0.001,
)
runtime.training.number_of_workers = STAGE2_DATALOADER_WORKERS
runtime.training.deterministic = not FAST_NONDETERMINISTIC
print(runtime.as_dict())

preflight_environment(runtime, require_dataset=True, require_training_dependencies=True)


In [ ]:
from puma.training.stage2_v132 import ensure_v132_split, phase_bounds
from puma.pipeline.oof import validate_full_oof

split_info = ensure_v132_split(runtime, force=False, val_fraction=0.20, seed=2026, check_sources=True)
validate_full_oof(runtime)

print('Split hash:', split_info['split_hash'])
print('Train/Val ROI:', len(split_info['train_roi_indices']), len(split_info['val_roi_indices']))
print('Case leakage:', split_info['diagnostics']['case_leakage_count'])
print('GT_POS / OOF_POS / OOF_ALL epochs:', phase_bounds(STAGE2_EPOCHS))


In [ ]:
from puma.runtime import resolve_hf_token
from puma.models.stage2 import ensure_stage2_pretrained_checkpoints

HF_TOKEN = resolve_hf_token()
checkpoint_summary = ensure_stage2_pretrained_checkpoints(
    PROJECT_DIR, hf_token=HF_TOKEN, pfm_keys=('uni2_h',)
)
print(checkpoint_summary['checkpoint_file'])


In [ ]:
from puma.stage2.catalog import VERSION132_EXPERIMENTS, VERSION132_EXPERIMENT_PURPOSE

STAGE2_EXPERIMENTS_TO_RUN = (
    VERSION132_EXPERIMENTS if STAGE2_EPOCHS == 50 else (WINNER_EXPERIMENT,)
)
for name in STAGE2_EXPERIMENTS_TO_RUN:
    print(f'- {name}: {VERSION132_EXPERIMENT_PURPOSE[name]}')


In [ ]:
from puma.pipeline.experiments_v132 import run_stage2_v132_program, aggregate_v132_results

stage2_summary = run_stage2_v132_program(
    runtime,
    hf_token=HF_TOKEN,
    experiments_to_run=STAGE2_EXPERIMENTS_TO_RUN,
)
ranking = aggregate_v132_results(runtime, STAGE2_EXPERIMENTS_TO_RUN)
if not ranking.empty:
    sort_cols = [c for c in ('macro_f1', 'conditional_type_macro_f1_present', 'reject_f1')
                 if c in ranking.columns]
    display(ranking.sort_values(sort_cols, ascending=False, na_position='last'))
stage2_summary


In [ ]:
# Lock only after reviewing results.
LOCK_WINNER = False
SELECTED_EXPERIMENT = WINNER_EXPERIMENT

if LOCK_WINNER:
    from puma.pipeline.experiments_v132 import lock_v132_winner

    locked = lock_v132_winner(
        runtime,
        selected_experiment=SELECTED_EXPERIMENT,
        candidate_experiments=tuple(STAGE2_EXPERIMENTS_TO_RUN),
    )
    print('Locked:', locked['selected_experiment'])
    print('Recommended final epochs:', locked['deployment']['recommended_final_epochs'])
